In [1]:
from pathlib import Path
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Draw
from openpyxl import load_workbook
from openpyxl.drawing.image import Image as XLImage

In [4]:
# =========================
# Settings
# =========================
input_file = "DataProcessing_43504_ASMS_DL2_DRplates.xlsx"
output_file = "compounds_with_structures.xlsx"
image_dir = Path("compound_images")

image_dir.mkdir(exist_ok=True)

In [ ]:
# =========================
# Read Excel
# =========================
df = pd.read_excel(input_file, sheet_name="Curves&Values")

required_columns = ["Compound", "SMILES",  "IC50 (uM)"]

missing = [col for col in required_columns if col not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

In [ ]:
# =========================
# Generate structure images
# =========================
image_paths = []

for i, row in df.iterrows():

    smiles = str(row["SMILES"]).strip()
    name = str(row["Compound"]).strip()
    ic50 = row["IC50 (uM)"]
    

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        print(f"WARNING: Invalid SMILES for {name}: {smiles}")
        image_paths.append(None)
        continue

    # Make filename safe
    safe_name = "".join(
        c if c.isalnum() or c in "-_." else "_"
        for c in name
    )

    image_path = image_dir / f"{i+1:04d}_{safe_name}.png"

    # Draw molecule
    img = Draw.MolToImage(
        mol,
        size=(500, 400)
    )

    img.save(image_path)

    image_paths.append(image_path)



In [ ]:
# =========================
# Add image references to dataframe
# =========================
df["Structure_Image"] = [
    str(p) if p else ""
    for p in image_paths
]


# Save basic dataframe first
df.to_excel(output_file, index=False)


In [10]:
# =========================
# Insert images into Excel
# =========================
wb = load_workbook(output_file)
ws = wb.active

# Add image column
image_column = ws.max_column + 1
ws.cell(row=1, column=image_column, value="Structure")

# Find column positions
name_col = df.columns.get_loc("Compound") + 1
ic50_col = df.columns.get_loc("IC50 (uM)") + 1

for excel_row, image_path in enumerate(image_paths, start=2):

    if image_path is None:
        continue

    img = XLImage(str(image_path))

    # Resize image for Excel
    img.width = 250
    img.height = 200

    cell = ws.cell(row=excel_row, column=image_column)
    cell.value = None

    ws.add_image(img, cell.coordinate)

    # Make row tall enough for image
    ws.row_dimensions[excel_row].height = 155


# Adjust column widths
ws.column_dimensions["A"].width = 25
ws.column_dimensions["B"].width = 30

# Structure image column
from openpyxl.utils import get_column_letter
image_col_letter = get_column_letter(image_column)
ws.column_dimensions[image_col_letter].width = 38


# Freeze header
ws.freeze_panes = "A2"

wb.save(output_file)